In [1]:
corpus = [
    "Silence is the sleep that nourishes wisdom.",
    "Science is a way of thinking much more than it is a body of knowledge.",
    "Spirituality is the art of keeping the inner fire alive.",
    "Socialism was given by Karl Marx to the working class.",
    "Success is never the final goal.",
    "Sanctions were imposed on Russia by the United States and the European Union."
]

In [35]:
import os
import re
import numpy as np
import pandas as pd
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.util import ngrams
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from nltk.stem import WordNetLemmatizer
from nltk.tag.hmm import HiddenMarkovModelTrainer
from gensim.models import Word2Vec
from nltk import pos_tag
import math
from collections import Counter


In [39]:
print("Text Preprocessing: ")
print("="*60)

#lowercasing, cleaning, tokenization, stopword removal, lemmatization, and basic nlp POS tagging
print("1. Lowercasing")
corpus_lower = [sen.lower() for sen in corpus]
print(corpus_lower)
print("2. Cleaning text")
corpus_cleaned = [re.sub(r'[^a-zA-Z\s]', '', sen) for sen in corpus_lower]
print(corpus_cleaned)
print("3. Tokenization")
corpus_tokenized = [word for sentence in [word_tokenize(sen) for sen in corpus_cleaned] for word in sentence]
print(corpus_tokenized)
print("4. Stopword Removal")
stop_words = set(stopwords.words('english'))
corpus_filtered = [word for word in corpus_tokenized if word not in stop_words]
print(corpus_filtered)
print("5. Lemmatization")
lemmatizer = WordNetLemmatizer()
corpus_lemmatized = [lemmatizer.lemmatize(word) for word in corpus_filtered]
print(corpus_lemmatized)
print("6. NLTK - Basic POS Tagging")
pos_tags = [pos_tag([word])[0] for word in corpus_lemmatized]
print(pos_tags)

Text Preprocessing: 
1. Lowercasing
['silence is the sleep that nourishes wisdom.', 'science is a way of thinking much more than it is a body of knowledge.', 'spirituality is the art of keeping the inner fire alive.', 'socialism was given by karl marx to the working class.', 'success is never the final goal.', 'sanctions were imposed on russia by the united states and the european union.']
2. Cleaning text
['silence is the sleep that nourishes wisdom', 'science is a way of thinking much more than it is a body of knowledge', 'spirituality is the art of keeping the inner fire alive', 'socialism was given by karl marx to the working class', 'success is never the final goal', 'sanctions were imposed on russia by the united states and the european union']
3. Tokenization
['silence', 'is', 'the', 'sleep', 'that', 'nourishes', 'wisdom', 'science', 'is', 'a', 'way', 'of', 'thinking', 'much', 'more', 'than', 'it', 'is', 'a', 'body', 'of', 'knowledge', 'spirituality', 'is', 'the', 'art', 'of', '

In [ ]:
print("N-grams and  its probability and perplexity measures")
print("="*60)
# Unigram
print("1. 1-grams: ")
unigram = list(ngrams(corpus_lemmatized, 1))
print(unigram)
# Bigrams
print("2. 2-grams: ")
bigram = list(ngrams(corpus_lemmatized, 2))
print(bigram)
#trigrams
print("3. 3-grams: ")
trigram = list(ngrams(corpus_lemmatized, 3))
print(trigram)

N-grams and  its probability and perplexity measures
1. 1-grams: 
[('silence',), ('sleep',), ('nourishes',), ('wisdom',), ('science',), ('way',), ('thinking',), ('much',), ('body',), ('knowledge',), ('spirituality',), ('art',), ('keeping',), ('inner',), ('fire',), ('alive',), ('socialism',), ('given',), ('karl',), ('marx',), ('working',), ('class',), ('success',), ('never',), ('final',), ('goal',), ('sanction',), ('imposed',), ('russia',), ('united',), ('state',), ('european',), ('union',)]
2. 2-grams: 
[('silence', 'sleep'), ('sleep', 'nourishes'), ('nourishes', 'wisdom'), ('wisdom', 'science'), ('science', 'way'), ('way', 'thinking'), ('thinking', 'much'), ('much', 'body'), ('body', 'knowledge'), ('knowledge', 'spirituality'), ('spirituality', 'art'), ('art', 'keeping'), ('keeping', 'inner'), ('inner', 'fire'), ('fire', 'alive'), ('alive', 'socialism'), ('socialism', 'given'), ('given', 'karl'), ('karl', 'marx'), ('marx', 'working'), ('working', 'class'), ('class', 'success'), ('succ

In [ ]:

# tokenize and add boundary markers (<s> start, </s> end)
train_tokens = []
for doc in corpus:
    tokens = ['<s>'] + word_tokenize(doc.lower()) + ['</s>']
    train_tokens.extend(tokens)


# calculate Unigram and Bigram Frequencies
unigram_counts = Counter(train_tokens)
bigram_counts = Counter(list(ngrams(train_tokens, 2)))

# vocabulary Size V (excluding boundary marker <s> from V count)
vocab = set(train_tokens) - {'<s>'}
V = len(vocab)


# probability Calculation Function with Laplace Smoothing
def get_bigram_prob(w1, w2):
    """Calculates P(w2 | w1) using Laplace (Add-1) Smoothing."""
    bigram_count = bigram_counts.get((w1, w2), 0)
    unigram_count = unigram_counts.get(w1, 0)
    
    # formula for probability of tokens or words: (C(w1, w2) + 1) / (C(w1) + V) [Total number of unique words in the vocabulary by counting the unique words in the training corpus]
    prob = (bigram_count + 1) / (unigram_count + V)
    return prob


# function to Compute Sentence Probability and Perplexity
def evaluate_sentence(test_sentence):
    test_words = ['<s>'] + word_tokenize(test_sentence.lower()) + ['</s>']
    test_bigrams = list(ngrams(test_words, 2))

    # total number of bigrams in the test sentence, sentence probability and log probability sum (for using in perplexity calculation)
    N = len(test_bigrams) 
    sentence_prob = 1.0
    log_prob_sum = 0.0

    print(f"\nEvaluating Sentence: \"{test_sentence}\"")
    print("-" * 55)
    print(f"{'Bigram (w1, w2)':<30} | {'Probability P(w2|w1)':<20}")
    print("-" * 55)

    for w1, w2 in test_bigrams:
        prob = get_bigram_prob(w1, w2)
        sentence_prob *= prob
        log_prob_sum += math.log(prob)  
        print(f"{str((w1, w2)):<30} | {prob:.6f}")

    # Calculating Perplexity 
    perplexity = math.exp(-log_prob_sum / N)

    print("-" * 55)
    print(f"Total Sentence Probability P(W) : {sentence_prob:.8e}")
    print(f"Perplexity (PP)                 : {perplexity:.4f}")
    return sentence_prob, perplexity


# Test Sentences
test_sent_1 = "science and silence nourishes knowledge"
test_sent_2 = "russia imposed karl marx way of thinking on european union"

evaluate_sentence(test_sent_1)
evaluate_sentence(test_sent_2)


Evaluating Sentence: "science and silence nourishes knowledge"
-------------------------------------------------------
Bigram (w1, w2)                | Probability P(w2|w1)
-------------------------------------------------------
('<s>', 'science')             | 0.036364
('science', 'and')             | 0.020000
('and', 'silence')             | 0.020000
('silence', 'nourishes')       | 0.020000
('nourishes', 'knowledge')     | 0.020000
('knowledge', '</s>')          | 0.020000
-------------------------------------------------------
Total Sentence Probability P(W) : 1.16363636e-10
Perplexity (PP)                 : 45.2582

Evaluating Sentence: "russia imposed karl marx way of thinking on european union"
-------------------------------------------------------
Bigram (w1, w2)                | Probability P(w2|w1)
-------------------------------------------------------
('<s>', 'russia')              | 0.018182
('russia', 'imposed')          | 0.020000
('imposed', 'karl')            | 0.020

(2.8643356643356646e-18, 39.338338055497736)

In [44]:
print("HMM TAGGING & LSI SVD CORRELATION")
print("="*60)

print("\n[A] HMM POS Tagging")

train_data = [
    [
        ('Silence', 'NOUN'), ('is', 'VERB'), ('the', 'DET'), ('sleep', 'NOUN'),
        ('that', 'PRON'), ('nourishes', 'VERB'), ('wisdom', 'NOUN'), ('.', 'PUNCT')
    ],
    [
        ('Science', 'NOUN'), ('is', 'VERB'), ('a', 'DET'), ('way', 'NOUN'),
        ('of', 'ADP'), ('thinking', 'NOUN'), ('much', 'ADV'), ('more', 'ADV'),
        ('than', 'ADP'), ('it', 'PRON'), ('is', 'VERB'), ('a', 'DET'),
        ('body', 'NOUN'), ('of', 'ADP'), ('knowledge', 'NOUN'), ('.', 'PUNCT')
    ],
    [
        ('Spirituality', 'NOUN'), ('is', 'VERB'), ('the', 'DET'), ('art', 'NOUN'),
        ('of', 'ADP'), ('keeping', 'VERB'), ('the', 'DET'), ('inner', 'ADJ'),
        ('fire', 'NOUN'), ('alive', 'ADJ'), ('.', 'PUNCT')
    ],
    [
        ('Socialism', 'NOUN'), ('was', 'VERB'), ('given', 'VERB'), ('by', 'ADP'),
        ('Karl', 'NOUN'), ('Marx', 'NOUN'), ('to', 'ADP'), ('the', 'DET'),
        ('working', 'ADJ'), ('class', 'NOUN'), ('.', 'PUNCT')
    ],
    [
        ('Success', 'NOUN'), ('is', 'VERB'), ('never', 'ADV'), ('the', 'DET'),
        ('final', 'ADJ'), ('goal', 'NOUN'), ('.', 'PUNCT')
    ],
    [
        ('Sanctions', 'NOUN'), ('were', 'VERB'), ('imposed', 'VERB'), ('on', 'ADP'),
        ('Russia', 'NOUN'), ('by', 'ADP'), ('the', 'DET'), ('United', 'ADJ'),
        ('States', 'NOUN'), ('and', 'CONJ'), ('the', 'DET'), ('European', 'ADJ'),
        ('Union', 'NOUN'), ('.', 'PUNCT')
    ]
]

# Train HMM Tagger on custom supervised data
hmm_trainer = HiddenMarkovModelTrainer()
hmm_tagger = hmm_trainer.train_supervised(train_data)

# Test sentence processing
test_sentence = "science and sleep nourishes knowledge"
test_tokens = word_tokenize(test_sentence)

# Perform HMM Tagging
hmm_predictions = hmm_tagger.tag(test_tokens)
print(f"Test Sentence: '{test_sentence}'")
print("HMM Tag Predictions:", hmm_predictions)


# LSI using SVD & Correlation
print("="*60)
print("\n[B] LSI via SVD & Correlation")
print("="*60)

# Generate TF-IDF Matrix
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)

# Perform SVD for LSI
n_components = 2
svd = TruncatedSVD(n_components=n_components, random_state=42)
corpus_lsi = svd.fit_transform(tfidf_matrix)

# Transform Test Sentence into 2D Vector Space
test_tfidf = vectorizer.transform([test_sentence])
test_lsi = svd.transform(test_tfidf)

print(f"\nTest Sentence Latent Vector (2D SVD Space):\n  {test_lsi[0]}")

# Compute Correlation with each Document
print("\nCorrelation of Test Sentence with Corpus Documents:")
for i, doc_vec in enumerate(corpus_lsi):
    # Calculate Correlation Matrix between vectors
    corr_matrix = np.corrcoef(test_lsi[0], doc_vec)
    correlation = corr_matrix[0, 1]
    
    print(f"  Doc {i+1}: '{corpus[i]}'")
    print(f"  --> Pearson Correlation: {correlation:.4f}\n")

HMM TAGGING & LSI SVD CORRELATION

[A] HMM POS Tagging
Test Sentence: 'science and sleep nourishes knowledge'
HMM Tag Predictions: [('science', 'NOUN'), ('and', 'NOUN'), ('sleep', 'NOUN'), ('nourishes', 'NOUN'), ('knowledge', 'NOUN')]

[B] LSI via SVD & Correlation

Test Sentence Latent Vector (2D SVD Space):
  [ 0.24977219 -0.06588461]

Correlation of Test Sentence with Corpus Documents:
  Doc 1: 'Silence is the sleep that nourishes wisdom.'
  --> Pearson Correlation: 1.0000

  Doc 2: 'Science is a way of thinking much more than it is a body of knowledge.'
  --> Pearson Correlation: 1.0000

  Doc 3: 'Spirituality is the art of keeping the inner fire alive.'
  --> Pearson Correlation: 1.0000

  Doc 4: 'Socialism was given by Karl Marx to the working class.'
  --> Pearson Correlation: -1.0000

  Doc 5: 'Success is never the final goal.'
  --> Pearson Correlation: 1.0000

  Doc 6: 'Sanctions were imposed on Russia by the United States and the European Union.'
  --> Pearson Correlation: -

C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\nltk\tag\hmm.py:333: RuntimeWarning: overflow encountered in cast
  X[i, j] = self._transitions[si].logprob(self._states[j])
C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\nltk\tag\hmm.py:335: RuntimeWarning: overflow encountered in cast
  O[i, k] = self._output_logprob(si, self._symbols[k])
C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\nltk\tag\hmm.py:331: RuntimeWarning: overflow encountered in cast
  P[i] = self._priors.logprob(si)
C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\nltk\tag\hmm.py:363: RuntimeWarning: overflow encountered in cast
  O[i, k] = self._output_logprob(si, self._s